# Statistics grouping across passes and interferograms

This notebook confirms that `StatsComputer.compute_input_stats` groups channels by slot-kind
(for example all interferogram phase channels share one strategy and one fitted scale), then
fits a per-channel location and scale. We inspect the resulting `ChannelStats` and confirm that
channels in the same group receive identical statistics.

Modules exercised:

- `pipelines.dataset_pipeline.normalization.StatsComputer`
- `configuration.dataset_config.InputConfig`

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path("../..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy             as np
import matplotlib.pyplot as plt

import matplotlib
matplotlib.rcParams["figure.dpi"] = 110
matplotlib.rcParams["image.interpolation"] = "nearest"

SEED = 0
np.random.seed(SEED)

try:
    import torch
    torch.manual_seed(SEED)
except Exception:
    pass

print("repo root on path:", REPO_ROOT)



## Dataset with several passes and interferograms

We build a stack large enough to fit meaningful statistics, with magnitude+angle primary,
real+imag secondaries and angle-only interferograms.


In [ ]:
from tools.monitoring.logger                            import Logger
from configuration.dataset_config            import InputConfig, OutputConfig
from tools.data.representation            import Representation
from pipelines.dataset_pipeline.spatial      import Patcher
from pipelines.dataset_pipeline.datasets     import PatchDataset
from pipelines.dataset_pipeline.normalization import StatsComputer

logger           = Logger(log_dir="/tmp/dataset_nb_logs", name="nb07", level="WARNING")
n_secondaries    = 2
n_interferograms = 4
n_passes         = 1 + n_secondaries + n_interferograms
H, W             = 48, 48

rng   = np.random.default_rng(SEED)
stack = (rng.standard_normal((n_passes, H, W)) + 1j * rng.standard_normal((n_passes, H, W))).astype(np.complex64)
gt    = rng.random((3, H, W)).astype(np.float32)

input_config = InputConfig(
    use_primary                   = True,
    primary_representation        = Representation.MAG_ANGLE,
    use_secondaries               = True,
    secondaries_representation    = Representation.REAL_IMAG,
    use_interferograms            = True,
    interferograms_representation = Representation.ANGLE_ONLY,
    use_dem                       = False,
)

patcher = Patcher.build(spatial_size=(H, W), patch_size=(24, 24), stride=12)
dataset = PatchDataset(inputs=stack, gt_parameters=gt, grid=patcher, input_config=input_config,
                       output_config=OutputConfig(), split_name="train",
                       n_secondaries=n_secondaries, n_interferograms=n_interferograms, n_gaussians=1)
print("patches:", len(dataset), " input channels:", dataset.input_channels)



## Fit input statistics

`compute_input_stats` collects sampled values per group and fits one (loc, scale) per group,
broadcast to every channel in that group.


In [ ]:
stats = StatsComputer.compute_input_stats(
    dataset          = dataset,
    logger           = logger,
    input_config     = input_config,
    n_secondaries    = n_secondaries,
    n_interferograms = n_interferograms,
    num_workers      = 0,
    batch_size       = 16,
)

cs    = stats.input_stats
keys  = input_config.channel_group_keys(n_secondaries, n_interferograms)
for c in range(cs.n_channels):
    print(f"ch {c:2d}  {keys[c]:16s}  loc={cs.loc[c]:+.4f}  scale={cs.scale[c]:.4f}  method={cs.strategies[c].norm_method.value}")



## Confirm grouping

Channels sharing a slot-kind should have identical loc and scale. We plot loc and scale per
channel coloured by group.


In [ ]:
unique_keys = list(dict.fromkeys(keys))
color_map   = {k: f"C{i}" for i, k in enumerate(unique_keys)}
colors      = [color_map[k] for k in keys]

fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
axes[0].bar(range(cs.n_channels), cs.loc,   color=colors)
axes[0].set_title("loc per channel"); axes[0].set_xlabel("channel")
axes[1].bar(range(cs.n_channels), cs.scale, color=colors)
axes[1].set_title("scale per channel"); axes[1].set_xlabel("channel")

handles = [plt.Rectangle((0, 0), 1, 1, color=color_map[k]) for k in unique_keys]
axes[1].legend(handles, unique_keys, loc="upper right", fontsize=8)
plt.tight_layout()
plt.show()


In [ ]:
for k in unique_keys:
    members = [i for i, key in enumerate(keys) if key == k]
    locs    = {round(cs.loc[i], 6)   for i in members}
    scales  = {round(cs.scale[i], 6) for i in members}
    print(f"{k:16s}  channels={members}  identical_loc={len(locs) == 1}  identical_scale={len(scales) == 1}")


## Expected visual outcome

Bars of the same colour (same slot-kind group) should have identical heights for both loc and
scale, demonstrating that statistics are shared within a group and fitted independently across
groups. The final check should print `identical_loc=True` and `identical_scale=True` for every
group. Phase groups should show loc 0 and scale pi.